In [ ]:
import pandas as pd
import sqlite3

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

paths = {
    "customers": PROJECT_ROOT / "data" / "olist_customers_dataset.xlsx",
    "geolocation": PROJECT_ROOT / "data" / "olist_geolocation_dataset.xlsx",
    "order_items": PROJECT_ROOT / "data" / "olist_order_items_dataset.xlsx",
    "order_payments": PROJECT_ROOT / "data" / "olist_order_payments_dataset.xlsx",
    "order_reviews": PROJECT_ROOT / "data" / "olist_order_reviews_dataset.xlsx",
    "orders": PROJECT_ROOT / "data" / "olist_orders_dataset.xlsx",
    "products": PROJECT_ROOT / "data" / "olist_products_dataset.xlsx",
    "sellers": PROJECT_ROOT / "data" / "olist_sellers_dataset.xlsx",
    "category_translation": PROJECT_ROOT / "data" / "product_category_name_translation.xlsx",
}

dataframes = {}


In [ ]:
conn = sqlite3.connect(":memory:")

In [ ]:
dataframes = {name: pd.read_excel(path) for name, path in paths.items()}

In [ ]:
from datetime import datetime

# Some Excel exports auto-format decimal monetary values as dates.
# Example: 26.08 can become 2026-08-26. Convert those cells back to numeric values.
def excel_date_number_to_float(value):
    if isinstance(value, datetime):
        if value.year == 2026:
            return value.day + value.month / 100
        return (value.year - 2000) + value.month / 100
    return value

numeric_columns = {
    "order_items": ["price", "freight_value"],
    "order_payments": ["payment_value"],
    "products": [
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
    ],
    "geolocation": ["geolocation_lat", "geolocation_lng"],
}

for table_name, columns in numeric_columns.items():
    for column in columns:
        if column in dataframes[table_name].columns:
            dataframes[table_name][column] = (
                dataframes[table_name][column]
                .map(excel_date_number_to_float)
                .pipe(pd.to_numeric, errors="coerce")
            )


In [ ]:
for name, path in paths.items():
    dataframes[name].to_sql(name, conn, index=False, if_exists="replace")


In [ ]:
print('Table names:')
for name, df in dataframes.items():
    print(name, df.columns)

In [ ]:
# Relationships between tables:

# orders.customer_id → customers.customer_id
# order_items.order_id → orders.order_id
# order_items.product_id → products.product_id
# order_items.seller_id → sellers.seller_id
# order_payments.order_id → orders.order_id
# order_reviews.order_id → orders.order_id
# products.product_category_name → category_translation.product_category_name

## Дослідницький Аналіз Данних

### Блок 1 — Загальний огляд бізнесу
1. Скільки всього замовлень у датасеті?
2. Скільки унікальних клієнтів?
3. Скільки продавців працює на платформі?
4. Скільки унікальних товарів продається?
5. Який середній чек замовлення?

### Блок 2 — Динаміка продажів
6. Як змінювалась кількість замовлень по місяцях?
7. Чи є сезонність у продажах?
8. Який місяць має найбільше продажів?
9. Як змінюється GMV (Gross Merchandise Value) з часом?

### Блок 3 — Поведінка клієнтів
10. Скільки замовлень робить середній клієнт?
11. Який % клієнтів роблять повторні покупки?
12. Який середній час між замовленнями?
13. Який Customer Lifetime Value (CLV)?

### Блок 4 — Аналіз платежів
14. Які способи оплати найпопулярніші?
15. Який середній чек по кожному способу оплати?
16. Який дохід по кожному способу оплати?

### Блок 5 — Відгуки клієнтів
17. Який середній рейтинг замовлень?
18. Яка частка негативних відгуків (1–2)?
19. Чи впливає час доставки на рейтинг?
20. Які категорії товарів отримують найгірші відгуки?

### Блок 6 — Логістика
21. Скільки в середньому триває доставка?
22. Чи є затримки доставки?
23. Які штати мають найбільші затримки?
24. Чи впливають затримки на рейтинг?

### Блок 7 — Аналіз продавців
25. Скільки замовлень у середнього продавця?
26. Хто топ-10 продавців за кількістю продажів?
27. Чи є залежність між рейтингом і продавцем?
28. Які продавці мають найбільше негативних відгуків?

### Блок 8 — Аналіз товарів
29. Які категорії товарів продаються найкраще?
30. Які категорії генерують найбільший дохід?
31. Які категорії мають найгірші відгуки?
32. Які категорії мають найдовшу доставку?

### Блок 9 — Географічний аналіз
33. В яких штатах найбільше покупців?
34. В яких штатах найбільше продавців?
35. Чи є різниця в середньому чеку між регіонами?
36. Чи є залежність між відстанню та часом доставки?

### Блок 10 — Ризики для бізнесу
37. Яка частка скасованих замовлень?
38. Чи пов’язані скасування з певними продавцями?
39. Чи впливає довга доставка на скасування?
40. Які категорії мають найбільше проблем?

## Exploratory Data Analysis

### Block 1 — General Business Overview
1. How many total orders are there in the dataset?
2. How many unique customers?
3. How many sellers work on the platform?
4. How many unique products are sold?
5. What is the average order receipt?

### Block 2 — Sales Dynamics
6. How did the number of orders change by month?
7. Is there seasonality in sales?
8. Which month has the most sales?
9. How does GMV (Gross Merchandise Value) change over time?

### Block 3 — Customer Behavior
10. How many orders does the average customer make?
11. What % of customers make repeat purchases?
12. What is the average time between orders?
13. What is the Customer Lifetime Value (CLV)?

### Block 4 — Payment Analysis
14. What are the most popular payment methods?
15. What is the average check for each payment method?
16. What is the revenue for each payment method?

### Block 5 — Customer Reviews
17. What is the average order rating?
18. What is the proportion of negative reviews (1–2)?
19. Does delivery time affect the rating?
20. Which product categories receive the worst reviews?

### Block 6 — Logistics
21. How long does delivery take on average?
22. Are there delivery delays?
23. Which states have the longest delays?
24. Do delays affect the rating?

### Block 7 — Seller Analysis
25. How many orders does the average seller have?
26. Who are the top 10 sellers by sales?
27. Is there a relationship between rating and seller?
28. Which sellers have the most negative reviews?

### Block 8 — Product Analysis
29. Which product categories sell the best?
30. Which categories generate the most revenue?
31. Which categories have the worst reviews?
32. Which categories have the longest shipping times?

### Block 9 — Geographic Analysis
33. Which states have the most buyers?
34. Which states have the most sellers?
35. Is there a difference in average check between regions?
36. Is there a relationship between distance and shipping time?

### Block 10 — Business Risks
37. What proportion of orders are canceled?
38. Are cancellations tied to specific sellers?
39. Does long shipping affect cancellations?
40. Which categories have the most problems

## Блок 1 — Загальний огляд бізнесу
1. Скільки всього замовлень у датасеті?\
2. Скільки унікальних клієнтів?
3. Скільки продавців працює на платформі?
4. Скільки унікальних товарів продається?
5. Який середній чек замовлення?

In [ ]:
# 1. Скільки всього замовлень у датасеті?
query = """
SELECT
    COUNT(DISTINCT(order_id))  AS cnt_orders
FROM
    orders
WHERE order_id IS NOT NULL
LIMIT 10;
"""
pd.read_sql_query(query, conn)

In [ ]:
# 2. Скільки унікальних клієнтів ?
query = """
SELECT
    COUNT(DISTINCT(customer_id)) AS uniq_customer
FROM
    orders
WHERE customer_id IS NOT NULL
LIMIT 10;
"""
pd.read_sql_query(query, conn)

In [ ]:
# 3. Скільки продавців працює на платформі?
query = """
SELECT
    COUNT(DISTINCT(seller_id)) AS cnt_sellers
FROM
    sellers
WHERE seller_id IS NOT NULL
LIMIT 10;
"""
pd.read_sql_query(query, conn)

In [ ]:
# 4. Скільки унікальних товарів продається?
query = """
SELECT
    COUNT(DISTINCT product_id) AS unique_products_sold
FROM order_items
WHERE product_id IS NOT NULL;
"""
pd.read_sql_query(query, conn)


In [ ]:
# 5. Який середній чек замовлення?
query = """
SELECT
    ROUND(AVG(order_total), 2) AS avg_order_value
FROM (
    SELECT
        order_id,
        SUM(payment_value) AS order_total
    FROM order_payments
    WHERE payment_value IS NOT NULL
    GROUP BY order_id
);
"""
pd.read_sql_query(query, conn)


## Блок 2 — Динаміка продажів
6. Як змінювалась кількість замовлень по місяцях?
7. Чи є сезонність у продажах?
8. Який місяць має найбільше продажів?
9. Як змінюється GMV (Gross Merchandise Value) з часом?

In [ ]:
# 6. Як змінювалась кількість замовлень по місяцях?
query = """
SELECT
    strftime('%Y-%m', order_purchase_timestamp) AS order_month,
    COUNT(*) AS orders_count
FROM
    orders
GROUP BY order_month 
ORDER BY order_month;
"""
pd.read_sql_query(query, conn)

### Висновок:
Кількість замовлень швидко зростала протягом 2017 року, збільшившись майже вдесятеро з січня по листопад.
Чіткий пік спостерігається в листопаді 2017 року, ймовірно, завдяки розпродажам у Чорну п'ятницю.
У 2018 році платформа досягла стабільного рівня попиту, а щомісячна кількість замовлень коливалася від 6500 до 7200.
Дані за вересень та жовтень 2018 року видаються неповними та не повинні враховуватися в аналізі тенденцій.

In [ ]:
# 7. Чи є сезонність у продажах?
query = """
SELECT
    substr(year_month, 6, 2) AS month,
    ROUND(AVG(month_orders),2) AS avg_orders
FROM
(
    SELECT
        strftime('%Y-%m', order_purchase_timestamp) AS year_month,
        COUNT(*) AS month_orders
    FROM orders
    GROUP BY year_month
    HAVING COUNT(*) > 1000
)
GROUP BY month
ORDER BY month;
"""
pd.read_sql_query(query, conn)

### Висновок: 
1) Аналіз виявляє сезонні закономірності в продажах. 
2) Обсяги замовлень досягають піку в листопаді, ймовірно, через акції Чорної п'ятниці, і залишаються відносно високими в грудні через святкові покупки. 
3) Продажі, як правило, нижчі в лютому та вересні, тоді як середина року демонструє відносно стабільний попит.

In [ ]:
# 8. Який місяць має найбільше продажів?
query = """
SELECT
    substr(year_month, 6, 2) AS month,
    ROUND(AVG(month_orders),2) AS avg_orders
FROM
(
    SELECT
        strftime('%Y-%m', order_purchase_timestamp) AS year_month,
        COUNT(*) AS month_orders
    FROM orders
    GROUP BY year_month
    HAVING COUNT(*) > 1000
)
GROUP BY month
ORDER BY avg_orders DESC
LIMIT 1;
"""
pd.read_sql_query(query, conn)

### Висновок:
Обсяги замовлень досягають піку в листопаді, ймовірно, через акції Чорної п'ятниці.  


In [ ]:
# 9. Як змінюється GMV (Gross Merchandise Value) з часом?
query = """
SELECT 
    strftime('%Y-%m', o.order_purchase_timestamp) AS year_month,
    ROUND(SUM(oi.price + oi.freight_value),2) AS gmv
    /* Сума GMV */
FROM
    orders o
JOIN order_items oi
ON o.order_id = oi.order_id
GROUP BY year_month
ORDER BY year_month; 
"""
pd.read_sql_query(query, conn)

### Висновок:
1) Жовтень 2016 року був не поганий як для почтаку роботи маркетплейса
2) Можно спостерігати дуже сильне зростання в 2017. GMV росте майже кожен місяць ~15х зростання за рік. 
2) Можно спостерігати передсезонну просадку в вересні 2017 року
3) Пік припадає на листопад 2017 року, ймовірна причина Black Friday. 

## Блок 3 — Поведінка клієнтів
10. Скільки замовлень робить середній клієнт?
11. Який % клієнтів роблять повторні покупки?
12. Який середній час між замовленнями?
13. Який Customer Lifetime Value (CLV)?

In [ ]:
# 10. Скільки замовлень робить середній клієнт?
query = """
SELECT
    ROUND(AVG(cnt_orders), 2) AS avg_ord_per_customer
FROM
(
SELECT
    customer_unique_id,
    COUNT(DISTINCT order_id) AS cnt_orders -- Кількість замовлень на кожного клієнта
FROM customers c
JOIN orders o
ON c.customer_id = o.customer_id
GROUP BY customer_unique_id
)
"""
pd.read_sql_query(query, conn)


### Висновок:
Середня кількість замовлень на одного клієнта становить приблизно 1,03, що свідчить про те, що більшість клієнтів розміщують лише одне замовлення на платформі і про відносно низький рівень утримання клієнтів.

In [ ]:
# 11. Який % клієнтів роблять повторні покупки?
query = """
SELECT
    ROUND(
        100.0 * SUM(CASE WHEN cnt_orders > 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS repeat_purchase_rate
FROM
(
SELECT
    customer_unique_id,
    COUNT(DISTINCT order_id) AS cnt_orders
FROM customers c
JOIN orders o 
ON c.customer_id = o.customer_id
GROUP BY customer_unique_id
)
"""
pd.read_sql_query(query, conn)



### Висновок: 
Рівень повторних покупок дуже низький, що свідчить про те, що більшість клієнтів здійснюють лише одну покупку на платформі. Це говорить про низький рівень утримання клієнтів.

In [ ]:
# 12. Який середній час між замовленнями?
query = """
SELECT
    ROUND(AVG(days_between_orders),2) AS avg_days_between_orders
FROM
(
SELECT
    customer_unique_id,
    julianday(order_purchase_timestamp) -
    julianday(LAG(order_purchase_timestamp)
        OVER (
            PARTITION BY customer_unique_id
            ORDER BY order_purchase_timestamp
        )
    ) AS days_between_orders
FROM customers c
JOIN orders o
ON c.customer_id = o.customer_id
)
WHERE days_between_orders IS NOT NULL;
""" 
pd.read_sql_query(query, conn)


### Висновок: 
Середній час між повторними покупками становить приблизно 78 днів, що свідчить про те, що постійні клієнти зазвичай розміщують наступне замовлення приблизно через два-три місяці. Це говорить про відносно довгий цикл покупки та підкреслює можливості для маркетингових стратегій утримання клієнтів.


In [ ]:
# 13. Який Customer Lifetime Value (CLV)?
query = """
SELECT
    AVG(customer_revenue) AS avg_clv
FROM (
    SELECT
        c.customer_unique_id, 
        SUM(oi.price + oi.freight_value) AS customer_revenue
    FROM
        orders o    
    JOIN customers c 
        ON o.customer_id = c.customer_id
    JOIN order_items oi 
         ON oi.order_id = o.order_id
    GROUP BY c.customer_unique_id
);
"""
pd.read_sql_query(query, conn)

### Висновок: 
1) Верхня межа витрати на маркетинг 100-300 якщо більше, на межі збитковості. 
2) CLV ≈ один чек
3) Бізнес залежить від нових клієнтів (Growth = Acquisition, а не retention)
- Постійсна залежність від реклами
- Низька лояльність
4) Найбільша можливість росту в збільшені повторних покупок

### Рекомендації:
1) Щоб покращіти CLV рекомендовано більше впроваджувати:
- email маркетинг
- персональні рекомендації
- бонуси за повторні покупки 
- підписки

## Блок 4 — Аналіз платежів
14. Які способи оплати найпопулярніші?
15. Який середній чек по кожному способу оплати?
16. Який дохід по кожному способу оплати?
 

In [ ]:
# 14. Які способи оплати найпопулярніші?
query = """
SELECT
    payment_type,
    COUNT(payment_type) AS cnt_type,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) AS percent 
FROM
    order_payments
WHERE payment_type IS NOT NULL
GROUP BY payment_type
ORDER BY cnt_type DESC;
"""
pd.read_sql_query(query, conn)

### Висновок:
1) Основний драйвер - credit card. Бізнес залежить від:
- платіжних шлюзів
- стабільних транзакцій
- комісій банків
2) boleto 
- це локальна специфіка Бразилії
- частина користувачів НЕ використовує карт
3) Voucher
- маркетинг вже працює
- є сегмент користвувачів, які реагують на знижки 
4) Debit card
- UX гірший ніж у credit card
- користвувачі не довіряють
- або просто не популярне

### Рекомендації:
1) Опитимізовувати credit card flow
- швидкість оплати
- мінімум відмов
- User Expirience (UX)
2) Розвивати альтернативи:
- boleto
- Apple Pay/ Google Pay
3) Використовувати voucher
- підвищувати retantion
- стимулювати повторні покупки


In [ ]:
# 15. Який середній чек по кожному способу оплати?
query = """
SELECT
    payment_type,
    ROUND(AVG(order_total), 2) AS avg_order_value
FROM (
    SELECT
        order_id,
        payment_type,
        SUM(payment_value) AS order_total
    FROM
        order_payments
    GROUP BY order_id, payment_type
)
GROUP BY payment_type
ORDER BY avg_order_value DESC;
 """
pd.read_sql_query(query, conn)

### Висновок:
1) У Voucher найвищій середній чек але не найбільший дохід це дуже важливо 
- знижки симулюють більше покупок
- клієнти "докуповують", щоб використовувати бонус
- маркетинг працює 
2) Credit Card - стабільний сегмент
3) Boleto s Debit - нижчій середній чек → дешевші покупки
- вірогідно більш чутливі до ціни клієнти
- менша платіжоспроможність
- або інший тип аудіторії
Ключовий бізнес-інсайт: Voucher збільшує середній чек → це інструмент для росту retention та revenue

### Рекомендації:
1) Маштабувати Voucher
- акції
- персональні знажки
- бонуси за великі покупки
2) Оптимізація Credit Card
- це ядро бізнесу
- тут найбільний обʼєм і дохід
3) Преревірити маржу:
- чи не зʼїдають знижки прибуток ?

In [ ]:
# 16. Який дохід по кожному способу оплати?
query = """
SELECT
    payment_type,
    ROUND(SUM(order_total), 2) AS revenue_pay_type
FROM (
    SELECT
        order_id,
        payment_type,
        SUM(payment_value) AS order_total
    FROM
        order_payments
    GROUP BY order_id, payment_type
)
GROUP BY payment_type
ORDER BY revenue_pay_type DESC;
 """
pd.read_sql_query(query, conn)

### Висновок:
Кредитні картки генерують більшу частину загального доходу, що робить їх основним методом оплати для бізнесу. Хоча оплата ваучерами має найвищу середню вартість замовлення, їхній загальний внесок у дохід є відносно невеликим через нижчу частоту використання. Це підкреслює розрив між розміром транзакції та загальним впливом на дохід. Boleto залишається важливим вторинним методом оплати, тоді як використання дебетових карток мінімальне.

## Блок 5 — Відгуки клієнтів
17. Який середній рейтинг замовлень?
18. Яка частка негативних відгуків (1–2)?
19. Чи впливає час доставки на рейтинг?
20. Які категорії товарів отримують найгірші відгуки?

In [ ]:
# 17. Який середній рейтинг замовлень?
query = """
SELECT
    ROUND(AVG(review_score), 2) AS avg_review_score
FROM 
    order_reviews
WHERE review_score IS NOT NULL;
 """
pd.read_sql_query(query, conn)

In [ ]:
# 18. Яка частка негативних відгуків (1–2)?
query = """
SELECT
    SUM(percent) AS part_reviews_score_1_and_2
FROM
    (
    SELECT
        review_score,
        COUNT(review_score) AS cnt_review_score,
        ROUND(100.0 * COUNT(*)/ SUM(COUNT(*))  OVER(), 2) AS percent
    FROM 
        order_reviews
    WHERE
       review_score IS NOT NULL
    GROUP BY review_score
    ORDER BY review_score
    )
WHERE review_score <= 2 
 """
pd.read_sql_query(query, conn)

### Висновок:
1) Головний інсайт: 
- Високий середній рейтинг маскує 15% негативу

### Рекомендації:
1) Аналіз негативу (ТОП пріорітет):
- категорії ()
- покращіти доставку
- payment ()
- фокус на (1-2) це найцінніші дані для покращення 
- зменшити негатив хочаб до 10%

In [ ]:
# 19. Чи впливає час доставки на рейтинг?
query = """
SELECT
    review_score,
    AVG(delivery_time) AS avg_delivery_time
FROM
    (
    SELECT
        julianday(order_delivered_customer_date) - julianday(order_approved_at) AS delivery_time,
        review_score  
    FROM 
        orders o
    JOIN order_reviews orr ON o.order_id = orr.order_id
    WHERE
        order_delivered_customer_date IS NOT NULL
        AND order_approved_at IS NOT NULL
    )
GROUP BY review_score
"""
pd.read_sql_query(query, conn)

### Висновок: 
1) Головний інсайт чим довша доставка тим нижче рейтинг
2) Доставка - один із ключових фаткорів задоволеності клієнтів

### Рекомендації: 
1) Оптимізовувати доставку
- логістика
- швидкість обрабки замовлень
- склади
2) Контролюваьти SLA (Service Level Agreement) 
- target: < 12 дні (після цього рейтинги різко ростуть)
3) Прогноз затримок
- попереджати клієнтів
- показувати реальний ETA (Estimated Time of Arrival) 
4) Пріоритезувати повільні регіони

In [ ]:
# 20. Які категорії товарів отримують найгірші відгуки?
query = """
SELECT
    ct.product_category_name_english,
    COUNT(*) AS review_count,
    ROUND(AVG(review_score), 2) AS avg_review_score
FROM
    products p
JOIN order_items oi
    ON p.product_id = oi.product_id
JOIN order_reviews orr
    ON oi.order_id = orr.order_id 
JOIN category_translation ct 
    ON p.product_category_name = ct.product_category_name
GROUP BY ct.product_category_name_english
HAVING COUNT(*) >= 30
ORDER BY avg_review_score ASC, review_count DESC
LIMIT 20;
"""
pd.read_sql_query(query, conn)


### Висновок:
1) Критичні категорії:
- office_furniture (Count: 1687, avg_review: 3.49)
- home_confort (Count: 435, avg_review: 3.83)
- audio (Count: 361, avg_review: 3.83)
Малий рейтинг + великий обʼєм = великий вплив на середню оцінку
2) Малі категорії (менше значення)
Шум а не проблема:
- diapers_and_hygiene (39 відгуків)
- party_supplies (43 відгуки)
3) Проблема сконцентрована в меблях і складних товарах:
- office_furniture
- furniture_*
- home_*
Ймовірна причина:
- Довга доставка товарів
- Пошкодження (Великі товари, Ризик транспортування)
- Очікування VS реальність (фото != реальність, Складність товарів)

### Рекомендації:
1) Фокус на office_furniture (Це найбільша точка болю)
- аналіз доставки
- аналіз поверненя
- аналіз review текстів
2) Покращіти логістику
- зменшити delivery_time
3) Контент товарі
- більше фото
- точні розміри
- відео/інструкції 
4) Сегментуванняч проблеми
- перевірити кореляцію між часом достави, категорією, оцінкою, кількітсь оцінок. 


## Блок 6 — Логістика
21. Скільки в середньому триває доставка?
22. Чи є затримки доставки?
23. Які штати мають найбільші затримки?
24. Чи впливають затримки на рейтинг?

In [ ]:
# 21. Скільки в середньому триває доставка?
query = """
SELECT
    AVG(julianday(order_delivered_customer_date) - julianday(order_approved_at)) AS delivery_time
FROM
    orders
"""
pd.read_sql_query(query, conn)



In [ ]:
# 22. Чи є затримки доставки?
query = """
SELECT
    ROUND(
        100.0 * SUM(
            CASE 
                WHEN julianday(order_delivered_customer_date)
                    > julianday(order_estimated_delivery_date)
                THEN 1 ELSE 0
            END
        ) / COUNT(*), 2
    ) AS delayed_percent
FROM
    orders
WHERE 
    order_delivered_customer_date IS NOT NULL
    AND order_estimated_delivery_date IS NOT NULL
"""
pd.read_sql_query(query, conn)


### Висновок:
1) 8.1 % замолень були доставлені невчасно

In [ ]:
# 23. Які штати мають найбільші затримки?
query = """
SELECT
    c.customer_state,
    COUNT(DISTINCT o.order_id) AS delivered_orders,
    SUM(CASE WHEN julianday(o.order_delivered_customer_date) > julianday(o.order_estimated_delivery_date) THEN 1 ELSE 0 END) AS delayed_orders,
    ROUND(100.0 * SUM(CASE WHEN julianday(o.order_delivered_customer_date) > julianday(o.order_estimated_delivery_date) THEN 1 ELSE 0 END) / COUNT(DISTINCT o.order_id), 2) AS delayed_rate_pct,
    ROUND(AVG(CASE WHEN julianday(o.order_delivered_customer_date) > julianday(o.order_estimated_delivery_date)
        THEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) END), 2) AS avg_delay_days_late_only
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL
GROUP BY c.customer_state
HAVING delivered_orders >= 100
ORDER BY delayed_rate_pct DESC, avg_delay_days_late_only DESC;
"""
pd.read_sql_query(query, conn)


### Висновок:
1) Кртичніть результату (Великий обʼєм + велика затримка)
2) Штати з дуже великою затримкаю але з малим обʼємо: AC, AP, RR(Шум або гегографічна складність)
Можливі причіни:
- віддаленість
- слабка логістика
- мало складів
3) Затримки не тільки в віддалених регіонах,
але і в ключових штатах з великим обʼємом замовлень.
Можливі причини:
- процесів 
- перевізників
- складів

### Рекомендації:
1) Пріорітезувати великі штати
- SP
- RJ
- MG
- RS
- PR
2) Оптимізувати логістику, наприклад:
- нові склади
- інші перевізники
- оптимізація маршрутів
3) Розділити SLA по регіонах. 
4) Створити різні ETA для кожного штату
5) необіцяти занадто швидку доставку
6) попереджати клієнтів
7) Перевірити перевізників

In [ ]:
# 24. Які штати мають найбільші затримки і в яких категоріях товарів ?
query = """
SELECT
    c.customer_state,
    (julianday(o.order_estimated_delivery_date)- julianday(o.order_delivered_customer_date)) AS avg_delay_days
FROM 
    orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE
    o.order_delivered_customer_date IS NOT NULL
    AND o.order_estimated_delivery_date IS NOT NULL
    AND c.customer_state = 'SP'
GROUP BY c.customer_state
ORDER BY avg_delay_days DESC;
"""
pd.read_sql_query(query, conn)


## Блок 7 — Аналіз продавців
25. Скільки замовлень у середнього продавця?
26. Хто топ-10 продавців за кількістю продажів?
27. Чи є залежність між рейтингом і продавцем?
28. Які продавці мають найбільше негативних відгуків?

In [ ]:
# 25. Скільки замовлень у середнього продавця?
query = """
SELECT
    ROUND(AVG(orders_count), 2) AS avg_orders_per_seller,
    ROUND(AVG(items_count), 2) AS avg_items_per_seller
FROM (
    SELECT
        seller_id,
        COUNT(DISTINCT order_id) AS orders_count,
        COUNT(*) AS items_count
    FROM order_items
    GROUP BY seller_id
);
"""
pd.read_sql_query(query, conn)


In [ ]:
# 26. Хто топ-10 продавців за кількістю продажів?
query = """
SELECT
    s.seller_id,
    COUNT(*) AS count_orders
FROM
    order_items oi
JOIN sellers s ON oi.seller_id = s.seller_id
GROUP BY s.seller_id
ORDER BY count_orders DESC
LIMIT 10;
"""
pd.read_sql_query(query, conn)


In [ ]:
# 27. Чи є залежність між рейтингом і продавцем?
query = """
SELECT
    MIN(avg_reviews_score) AS min_avg,
    MAX(avg_reviews_score) AS max_avg,
    MAX(avg_reviews_score) - MIN(avg_reviews_score) AS difference_avg
FROM
(
SELECT
    s.seller_id,
    COUNT(DISTINCT oi.order_id) AS count_orders,
    ROUND(AVG(orr.review_score), 3) AS avg_reviews_score,
    COUNT(orr.review_score) AS count_reviews
FROM
    order_items oi
JOIN sellers s ON oi.seller_id = s.seller_id
JOIN order_reviews orr ON orr.order_id = oi.order_id
GROUP BY s.seller_id
HAVING COUNT(DISTINCT oi.order_id) > 50
ORDER BY avg_reviews_score ASC
)
"""
pd.read_sql_query(query, conn)



### Висконовок:
1) Є залежність між продавцями і рейтингом

### Рекомнедації:
1) Ввести рейтинг продавцівє
2) Фільтрувати поганих продавців
3) Ввести SLA для продавців
4) Показувати "Top Sellers"



In [ ]:
# 28. Які продавці мають найбільше негативних відгуків?
query = """

SELECT
    s.seller_id,
    COUNT(DISTINCT CASE WHEN orr.review_score <= 2  THEN orr.review_id END) AS count_bad_reviews,
    COUNT(DISTINCT orr.review_id) AS total_reviews,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN orr.review_score <= 2  THEN orr.review_id END)/COUNT(DISTINCT orr.review_id),2) AS bad_review_rate
FROM
    order_items oi
JOIN sellers s ON oi.seller_id = s.seller_id
JOIN order_reviews orr ON orr.order_id = oi.order_id
JOIN orders o ON oi.order_id = o.order_id
GROUP BY s.seller_id
HAVING COUNT(DISTINCT oi.order_id) > 50
ORDER BY bad_review_rate DESC
LIMIT 20;
"""
pd.read_sql_query(query, conn)


### Висновок:
1) Є критично промблемні продавці, в яких кожен другий клієнт не задоволений
- такі продавці прямо шкодять бізнесу, репутації
2) Велика группа продавців в зоні ризику 30-40% негативу
3) Проблема не в "маленьких продавцях", а в якості
4) Поріг "здорового бізнесу" порушений bad_review_rate < 10-15%
- або проблема платформи
- або дуже слабкий контроль продавців

### Рекомендації: 
1) Ввести жорстку сегментацію продавців
2) Призупинити топ продавців з найвищим bad_review_rate
3) Зʼясувати причини поганих відгуків
- логістика
- якість товару
- відповідність опису
4) Вплинути на видачу (UX / продукт)
- знизити позиції продавців у пошуку
- позувати рейтинг
- додати індикатор warning (поганого продавця)
5) Побудувати систему моніторингу тенденцій покращенні / погіршення відгуків

## Блок 8 — Аналіз товарів
29. Які категорії товарів продаються найкраще?
30. Які категорії генерують найбільший дохід?
31. Які категорії мають найгірші відгуки?
32. Які категорії мають найдовшу доставку?

In [ ]:
# 29. Які категорії товарів продаються найкраще?
query = """
SELECT
    ct.product_category_name_english,
    COUNT(DISTINCT oi.order_id) AS count_order
FROM
    products p
JOIN order_items oi ON p.product_id = oi.product_id
JOIN category_translation ct ON p.product_category_name = ct.product_category_name
WHERE ct.product_category_name_english IS NOT NULL
GROUP BY ct.product_category_name_english
ORDER BY count_order DESC
LIMIT 20
"""

pd.read_sql_query(query, conn)


### Висновок:
1) ТОП категорії:
- bed_bath_table
- health_beauty
- sports_leisure
Це означає:
- масовий попит
- everyday товари
- "широкий ринок"
2) Cередній сегмент:
electronics → 2550 (нижче, ніж очікуєш)
computers_accessories → 6689 (вище)
- люди частіше купують дешевші доповнення, ніж дорогі товари
3) Нішеві категорії:
consoles_games → 1062
office_furniture → 1273
- вузький попит
- або дорогі товари
4) Люди купуть часто те, що потрібно регулярно

### Рекомендації: 
1) Перевірити моржинальність, можливо:
- bed_bath → багато продажів, але мало грошей
- electronics → мало продажів, але багато грошей
2) Інвестувати в ТОП категорії
3) Upsell стратегія
- якщо купують bed_bath_table → додати decor / furniture
4) Electronics - окрема стратегія (низька частота ≠ погано, це може дорівнювати зависока ціна, інша поведінка)
Можливі варіанти рішення:
- інший марктеинг
- довша sales funnel


In [ ]:
# 30. Які категорії генерують найбільший дохід?
query = """
SELECT
    ct.product_category_name_english,
    COUNT(DISTINCT oi.order_id) AS count_orders,
    ROUND(SUM(oi.price + oi.freight_value), 2) AS revenue,
    AVG(oi.price) AS avg_price
FROM
    products p
JOIN order_items oi ON p.product_id = oi.product_id
JOIN category_translation ct ON p.product_category_name = ct.product_category_name
WHERE ct.product_category_name_english IS NOT NULL
GROUP BY ct.product_category_name_english
ORDER BY revenue DESC
LIMIT 20
"""
pd.read_sql_query(query, conn)


### Висновок:
1) Гроші робляться на повсякденних товарах
2) Значно нижче revenue на electronics може означати:
- або низький попит
- або конкуренція
- або інша поведінка клієнтів
3) Аксесуари купують частіше ніж електроніку це класичний e-commerce патерн
4) Нішеві категорії:
consoles_games
office_furniture
- вузький попит 
- або дорогі покупки з низькою частотою

### Рекомендації: 
1) Фокус на high-valume категоріях
2) Upsell / cross-sell
- якщо купують bed_bath → пропонувати decor/ housewares
- якщо купують computers → пропонувати accessories
3) Для electronics потрібна окрема стратегія, не маштабувати blindly, а працювати з довірою, брендом, гарантіями
4) Перевірити маржинальність, можливо:
bed_bath → великий обʼєм, низька маржа
electronics → малий обʼєм, висока маржа
5) Supply chain 
-топ категорії = високицй попит, потрібно тримати stock і уникати out-of-stock


In [ ]:
# 31. Які категорії мають найгірші відгуки?
query = """
SELECT
    ct.product_category_name_english,
    ROUND(AVG(orr.review_score), 2) AS avg_review,
    COUNT(DISTINCT orr.review_id) AS count_score,
    COUNT(DISTINCT CASE WHEN orr.review_score <= 2 THEN orr.review_id END) AS count_bad_review,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN orr.review_score <= 2 THEN orr.review_id END)/ COUNT(DISTINCT orr.review_id), 2) AS percent_negative_review
FROM
    products p
JOIN order_items oi ON p.product_id = oi.product_id
JOIN category_translation ct ON p.product_category_name = ct.product_category_name
JOIN order_reviews orr ON orr.order_id = oi.order_id
GROUP BY ct.product_category_name_english 
HAVING COUNT(orr.review_score) > 50
ORDER BY percent_negative_review DESC
LIMIT 20;
"""

pd.read_sql_query(query, conn)


### Висновок:
1) Є критично проблемні категорії
fashion_male_clothing → 26%
office_furniture → 22.7%
audio → 22.1%
- це дуже багато
- проблема системна а не випадкова.
2) Найнебезпечніші категорії (найбільший ризик для бізнесу)
office_furniture:
1265 відгуків
22.7% негативу
- великий обʼєм високий негатив
3) Масові категорії теж "хворі"
bed_bath_table → 16.7% (9324 reviews)
furniture_decor → 16.6% (6404 reviews)
computers_accessories → 15.9% (6641 reviews)
- проблема не тільки в нішах - вона в core бізнесі

### Рекомендації: 
1) Пріорітет = high volume + high negative
топ-фокус на великы об'эми
2) Root cause analysis (RCA)
- доставка
- якість 
- очікування vs реальність
3) Контроль продавців
- часто проблеми не в категоріях а в конкретних продавцях
4) Product/ UX рішення
- показувати рейтинг
- показувати % негативу
- додати "real reviews"
5) Ризики для бізнесу
- падає retention
- падає довіра
- падає revenue


In [ ]:
# 32. Які категорії мають найдовшу доставку?
query = """
SELECT
    ct.product_category_name_english,
    ROUND(AVG(julianday(order_delivered_customer_date) - julianday(order_approved_at)), 2) AS avg_delivery_time,
    COUNT(DISTINCT o.order_id) AS total_orders
FROM
    products p
JOIN order_items oi ON p.product_id = oi.product_id
JOIN category_translation ct ON p.product_category_name = ct.product_category_name
JOIN orders o ON o.order_id = oi.order_id
WHERE o.order_delivered_customer_date IS NOT NULL
  AND o.order_approved_at IS NOT NULL
GROUP BY ct.product_category_name_english 
HAVING COUNT(DISTINCT o.order_id) > 50
ORDER BY avg_delivery_time DESC
LIMIT 20;
"""

pd.read_sql_query(query, conn)


### Висновок:
1) Важливий патерн (довга доставка):
- меблі
- великі товари
- складних у логістиці

### Рекомендації: 
1) Пріорітет №1 - office_furniture
- переврити логістику
- перевірити склади
- перевірити продавців
2) Оптимізувати high-volume категорії, фокус на:
- computer_accessories
- furniture_decor
- garden_tools
3) Розділити категорії по логістиці:
- accessories (fast delivery)
- furniture (slow delivery)
4) Очікування клієнта (UX). Якщодоставка довга:
- показувати реальний ETA
- попереджати
- додаваии tracking

## Блок 9 — Географічний аналіз
33. В яких штатах найбільше покупців?
34. В яких штатах найбільше продавців?
35. Чи є різниця в середньому чеку між регіонами?
36. Чи є залежність між відстанню та часом доставки?

In [ ]:
# 33. В яких штатах найбільше покупців?
query = """
SELECT
    customer_state,
    COUNT(DISTINCT customer_unique_id) AS count_customers
FROM
    customers
GROUP BY customer_state
ORDER BY count_customers DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)


### Висновок:
1) Домінує штат SP 
- головний ринок бізнесу (≈40% від топ-10)
2) Ядро бізнесу, основний попит (63к):
- SP → 40k  
- RJ → 12k  
- MG → 11k  
3) Далі різкий спад:
- RS → 5k  
- PR → 4.8k  
4) Довгий "хвіст": SC, BA, DF, ES, GO → ~2k–3k
- потенціал для росту
- другорядні ринки

### Рекомендації: 
1) SP - максимальний фокус
- маркетинг
- швидка доставка
- склади 
2) Маштабування RJ і MG
3) Розвивати середні штати RS, PR:
- тестувати рекламу
- покращіти логістику
4) Дослідтии малі ринки:
- чому малі ринки ?
    - доставка ?
    - маркетинг ? 
    - попит ?
5) Зменшити залежність від SP
- якщо падає SP - падає весь бізнес


In [ ]:
# 34. В яких штатах найбільше продавців?
query = """
SELECT
    seller_state,
    COUNT(DISTINCT seller_id) AS count_sellers
FROM
    sellers
GROUP BY seller_state
ORDER BY count_sellers DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)


### Висновок:
1) Абсолютний лідер SP
2) Дисбаланс у топ-штатах
- (RJ) ~72 customers на 1 seller
- (MG) ~46 customers на 1 seller
3) Слібкі ринки:
- BA ~172 customers на 1 seller (величезне opportunity)
4) Головний інсайт: ринок має сильний інсайт між попитом і пропозицією, у деяких штатах (RJ, BA, MG), спостерігається значний дефіцит продавців при високому попиті.

### Рекомендації: 
1) Розвивати seller base у дефіцитних регіонах:
- Фокус:
    - RJ
    - MG
    - BA
- Дії:
    - залучення нових продавців
    - зниження барʼєрів входу
    - впровадження бонусних програм
    - зниження комісій
2) Контроль конкурменції
- Проблеми:
    - багато продавців
    - висока конкуренція
- Дії:
    - ранжування
    - якість
    - рейтинг
3) Логістика
- Проблема
    - довга доставка 
    - більше негативу
- Рішення:
    - локальні склади
    - fulfillment center
4) Маркетинг у дефіцитних регіонах

In [ ]:
# 35. Чи є різниця в середньому чеку між регіонами?
query = """
SELECT
    c.customer_state,
    ROUND(AVG(order_total), 2) AS avg_order_value,
    COUNT(*) AS orders_count,
    COUNT(DISTINCT customer_unique_id) AS count_customers
FROM
(
    SELECT
        o.order_id,
        o.customer_id,
        SUM(oi.price + oi.freight_value) AS order_total
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY o.order_id, o.customer_id
) t
JOIN customers c ON t.customer_id = c.customer_id
GROUP BY c.customer_state
HAVING COUNT(*) > 100
ORDER BY avg_order_value DESC;
"""

pd.read_sql_query(query, conn)


### Висновок:
1) Різниця між регіонами- Критична TOP ≈ 2,3 (поведінка сильно відрізняється)
2) SP - масовий ринок з низьким чеком
3) Premium regoin: MS, DF, RS
4) Balanced region ідельні в: RJ, MG (націнніші регіони для бізнесу)
5) Слабкі регіони: MO, MA, PI (низька купівельна спроможність або слабкий ринок)

### Рекомендації: 
1) SP- підіймати середній чек (навіть 10% = величезний приріст), дії:
- bundles
- upsell
- cross-sell
2) RJ та MG - маштабувати, дії:
- більше реклами
- розширення асортименту
3) Преміум регіон - окрема стратегія: MS, DF. Дії: 
- преміум товари
4) Слабкі регіони - перевірити:
- логістику
- дохід населеня
- доступність товарів

In [ ]:
# 36. Чи є залежність між відстанню та часом доставки?
query = """
WITH geo AS (
    SELECT
        geolocation_zip_code_prefix,
        AVG(geolocation_lat) AS lat,
        AVG(geolocation_lng) AS lng
    FROM geolocation
    GROUP BY geolocation_zip_code_prefix
), order_distance AS (
    SELECT
        o.order_id,
        ABS(gc.lat - gs.lat) + ABS(gc.lng - gs.lng) AS distance_proxy,
        julianday(o.order_delivered_customer_date) - julianday(o.order_approved_at) AS delivery_days
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN sellers s ON oi.seller_id = s.seller_id
    JOIN geo gc ON c.customer_zip_code_prefix = gc.geolocation_zip_code_prefix
    JOIN geo gs ON s.seller_zip_code_prefix = gs.geolocation_zip_code_prefix
    WHERE o.order_delivered_customer_date IS NOT NULL
      AND o.order_approved_at IS NOT NULL
)
SELECT
    ROUND(AVG(distance_proxy), 4) AS avg_distance_proxy,
    ROUND(AVG(delivery_days), 2) AS avg_delivery_days
FROM order_distance;
"""
pd.read_sql_query(query, conn)


### Рекомендації: 
1) Оптимізувати логістику пріорітет №1 (12 днів = проблема), перевірити:
- час обробки
- час відправки
- затримки на складі
2) Зменшити "last mile"
- відкрити склади ближче до клієнтів
- використовувати локальних продавців
3) Сегментувати доставку
4) Покращіти UX

## Блок 10 — Ризики для бізнесу
37. Яка частка скасованих замовлень?
38. Чи пов’язані скасування з певними продавцями?
39. Чи впливає довга доставка на скасування?
40. Які категорії мають найбільше проблем?

In [ ]:
# 37. Яка частка скасованих замовлень?
query = """
SELECT
    COUNT(*) AS count_all,
    SUM(CASE WHEN order_status = 'canceled' THEN 1 ELSE 0 END) AS count_canceled,
    SUM(100.0 * CASE WHEN order_status = 'canceled' THEN 1 ELSE 0 END) / COUNT(*) AS percent_canceled
FROM
    orders
"""

pd.read_sql_query(query, conn)


In [ ]:
# 38. Чи пов’язані скасування з певними продавцями?
query = """
SELECT
    oi.seller_id,
    COUNT(DISTINCT CASE WHEN o.order_status = 'canceled' THEN o.order_id END) AS count_canceled
FROM
    order_items oi
JOIN orders o ON oi.order_id = o.order_id
GROUP BY oi.seller_id
HAVING count_canceled > 1
ORDER BY count_canceled DESC
LIMIT 20;
"""

pd.read_sql_query(query, conn)


In [ ]:
# 39. Чи впливає довга доставка на скасування?
query = """
SELECT
    CASE WHEN (julianday(order_estimated_delivery_date) - julianday(order_purchase_timestamp)) > 7 THEN 'long_delivery' ELSE 'short_delivery' END AS delivery_group,
    COUNT(*) AS total_orders,
    SUM(CASE WHEN order_status = 'canceled' THEN 1 ELSE 0 END) AS canceled_orders,
    ROUND(100.0 * SUM(CASE WHEN order_status = 'canceled' THEN 1 ELSE 0 END) / COUNT(*), 2) AS cancel_rate
FROM orders
GROUP BY delivery_group
"""

pd.read_sql_query(query, conn)


In [ ]:
# 40. Які категорії мають найбільше проблем?
query = """
SELECT
    pct.product_category_name_english AS category,
    COUNT(*) AS total_orders,
    SUM(CASE WHEN o.order_status = 'canceled' THEN 1 ELSE 0 END) AS canceled_orders,
    ROUND(100.0 * SUM(CASE WHEN o.order_status = 'canceled' THEN 1 ELSE 0 END)/ COUNT(*), 2) AS cancel_rate
FROM
    order_items oi
JOIN orders o ON oi.order_id = o.order_id
JOIN products p ON oi.product_id = p.product_id
JOIN category_translation pct ON p.product_category_name = pct.product_category_name
GROUP BY category
HAVING total_orders > 50
ORDER BY cancel_rate DESC
"""

pd.read_sql_query(query, conn)


In [ ]:
query = """
SELECT
    *
FROM orders o
JOIN order_items oi 
    ON o.order_id = oi.order_id
JOIN sellers s 
    ON oi.seller_id = s.seller_id
JOIN customers c 
    ON o.customer_id = c.customer_id
JOIN geolocation gc 
    ON c.customer_zip_code_prefix = gc.geolocation_zip_code_prefix
JOIN geolocation gs 
    ON s.seller_zip_code_prefix = gs.geolocation_zip_code_prefix
JOIN products p
    ON p.product_id = oi.product_id
JOIN order_payments op
    ON o.order_id = op.order_id
JOIN category_translation ct
    ON ct.product_category_name = p.product_category_name
LIMIT 5;
"""
pd.read_sql_query(query, conn)


In [ ]:
query = """
SELECT
    *
FROM products

"""
pd.read_sql_query(query, conn)
